#**LSTM_Training**

In [1]:
# Cell 1 — Mount Drive + imports
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

BASE    = "/content/drive/MyDrive/EyeMotionID"
MODELS  = os.path.join(BASE, "models/checkpoints")
PLOTS   = os.path.join(BASE, "results/plots")
os.makedirs(MODELS, exist_ok=True)
os.makedirs(PLOTS,  exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✓ Drive mounted")
print(f"✓ Device : {device}")
print(f"✓ PyTorch: {torch.__version__}")

Mounted at /content/drive
✓ Drive mounted
✓ Device : cpu
✓ PyTorch: 2.11.0+cpu


###Build CNN feature extractor

In [2]:
class CNNFeatureExtractor(nn.Module):
    """
    ResNet18 backbone with final layer removed.
    Input : (batch, 3, 224, 224)
    Output: (batch, 512)
    """
    def __init__(self):
        super(CNNFeatureExtractor, self).__init__()
        resnet = models.resnet18(weights='IMAGENET1K_V1')
        # remove final classification layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])

    def forward(self, x):
        x = self.features(x)          # (batch, 512, 1, 1)
        x = x.view(x.size(0), -1)     # (batch, 512)
        return x

# test CNN extractor
cnn = CNNFeatureExtractor().to(device)

# freeze all parameters
for param in cnn.parameters():
    param.requires_grad = False
cnn.eval()

# test with dummy input
dummy_img = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    cnn_out = cnn(dummy_img)

print("=== CNN Feature Extractor ===\n")
print(f"Input shape  : {dummy_img.shape}")
print(f"Output shape : {cnn_out.shape}")
print(f"Parameters   : {sum(p.numel() for p in cnn.parameters()):,} (all frozen)")
print("\n✓ CNN extractor ready")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 51.2MB/s]


=== CNN Feature Extractor ===

Input shape  : torch.Size([4, 3, 224, 224])
Output shape : torch.Size([4, 512])
Parameters   : 11,176,512 (all frozen)

✓ CNN extractor ready


###2-layer LSTM that processes sequences of CNN feature vectors.

In [3]:
class LSTMTemporalModel(nn.Module):
    """
    LSTM for temporal eye movement sequence modeling.
    Input : (batch, seq_len, input_size=512)
    Output: (batch, hidden_size=256)
    """
    def __init__(self, input_size=512, hidden_size=256,
                 num_layers=2, dropout=0.5):
        super(LSTMTemporalModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch, seq_len, 512)
        lstm_out, (hidden, _) = self.lstm(x)
        # use last hidden state
        out = self.dropout(hidden[-1])  # (batch, 256)
        return out

# test LSTM
lstm = LSTMTemporalModel().to(device)

dummy_seq = torch.randn(4, 10, 512).to(device)  # (batch=4, seq=10, features=512)
lstm_out  = lstm(dummy_seq)

print("=== LSTM Temporal Model ===\n")
print(f"Input shape  : {dummy_seq.shape}  (batch, seq_len, features)")
print(f"Output shape : {lstm_out.shape}   (batch, hidden_size)")
print(f"Parameters   : {sum(p.numel() for p in lstm.parameters()):,} (trainable)")
print("\n✓ LSTM model ready")

=== LSTM Temporal Model ===

Input shape  : torch.Size([4, 10, 512])  (batch, seq_len, features)
Output shape : torch.Size([4, 256])   (batch, hidden_size)
Parameters   : 1,314,816 (trainable)

✓ LSTM model ready


##Build complete EyeMotionID model

#####Combines CNN + LSTM + classifier into one unified model.

In [4]:
class EyeMotionID(nn.Module):
    """
    Complete EyeMotionID behavioral biometric model.

    Pipeline:
    Input (batch, seq_len, 3, 224, 224)
        → CNN  → (batch, seq_len, 512)
        → LSTM → (batch, 256)
        → FC   → (batch, n_users)

    Args:
        n_users    : number of users to identify
        seq_len    : number of frames per sequence
        hidden_size: LSTM hidden units
        num_layers : LSTM layers
        dropout    : dropout rate
    """
    def __init__(self, n_users=15, seq_len=10,
                 hidden_size=256, num_layers=2, dropout=0.5):
        super(EyeMotionID, self).__init__()
        self.seq_len = seq_len
        self.cnn     = CNNFeatureExtractor()

        # freeze CNN
        for param in self.cnn.parameters():
            param.requires_grad = False

        self.lstm       = LSTMTemporalModel(512, hidden_size, num_layers, dropout)
        self.classifier = nn.Linear(hidden_size, n_users)

    def forward(self, x):
        # x: (batch, seq_len, 3, 224, 224)
        B, T, C, H, W = x.shape

        # extract CNN features for each frame
        x = x.view(B * T, C, H, W)          # (B*T, 3, 224, 224)
        with torch.no_grad():
            x = self.cnn(x)                  # (B*T, 512)
        x = x.view(B, T, -1)                 # (B, T, 512)

        # temporal modeling
        x = self.lstm(x)                     # (B, 256)

        # classification
        x = self.classifier(x)               # (B, n_users)
        return x

# build model for 15 users (MPIIGaze)
model_15 = EyeMotionID(n_users=15).to(device)

print("=== EyeMotionID Model (15 users) ===\n")
total  = sum(p.numel() for p in model_15.parameters())
frozen = sum(p.numel() for p in model_15.parameters() if not p.requires_grad)
trainable = total - frozen
print(f"Total params    : {total:,}")
print(f"Frozen (CNN)    : {frozen:,}")
print(f"Trainable       : {trainable:,} ← only LSTM + classifier train")
print("\n✓ EyeMotionID model ready")

=== EyeMotionID Model (15 users) ===

Total params    : 12,495,183
Frozen (CNN)    : 11,176,512
Trainable       : 1,318,671 ← only LSTM + classifier train

✓ EyeMotionID model ready


###Passes a dummy batch of eye sequences through the complete model.

In [5]:
# test complete forward pass
# dummy input: batch=2, seq=10, channels=3, H=224, W=224
dummy_input = torch.randn(2, 10, 3, 224, 224).to(device)

model_15.eval()
with torch.no_grad():
    output = model_15(dummy_input)

print("=== Forward Pass Test ===\n")
print(f"Input  : {list(dummy_input.shape)}")
print(f"         (batch=2, frames=10, C=3, H=224, W=224)")
print()
print(f"Output : {list(output.shape)}")
print(f"         (batch=2, users=15)")
print()
print("Output values (raw logits):")
print(f"  Sample 1: {output[0].tolist()[:5]} ...")
print(f"  Sample 2: {output[1].tolist()[:5]} ...")
print()

# apply softmax to get probabilities
probs = torch.softmax(output, dim=1)
pred  = torch.argmax(probs, dim=1)
print(f"Predicted users : {pred.tolist()}")
print(f"Confidence      : {probs.max(dim=1).values.tolist()}")
print("\n✓ Forward pass working!")

=== Forward Pass Test ===

Input  : [2, 10, 3, 224, 224]
         (batch=2, frames=10, C=3, H=224, W=224)

Output : [2, 15]
         (batch=2, users=15)

Output values (raw logits):
  Sample 1: [0.0028536482714116573, -0.014868775382637978, -0.11283381283283234, 0.025456953793764114, 0.11807135492563248] ...
  Sample 2: [0.00460934080183506, -0.010467462241649628, -0.10935512185096741, 0.025753499940037727, 0.11601056903600693] ...

Predicted users : [4, 4]
Confidence      : [0.07576880604028702, 0.07565838098526001]

✓ Forward pass working!


###Define loss function and optimizer

In [7]:
# loss function
criterion = nn.CrossEntropyLoss()

# optimizer — only train LSTM + classifier (CNN is frozen)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_15.parameters()),
    lr=0.001,
    weight_decay=1e-4
)

# learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

print("=== Training Configuration ===\n")
print(f"Loss function : CrossEntropyLoss")
print(f"Optimizer     : Adam")
print(f"Learning rate : 0.001")
print(f"Weight decay  : 1e-4")
print(f"LR scheduler  : ReduceLROnPlateau (patience=3)")
print()

# test loss on dummy data
dummy_labels = torch.randint(0, 15, (2,)).to(device)
loss = criterion(output, dummy_labels)
print(f"Dummy loss    : {loss.item():.4f}")
print("\n✓ Loss + optimizer ready")

=== Training Configuration ===

Loss function : CrossEntropyLoss
Optimizer     : Adam
Learning rate : 0.001
Weight decay  : 1e-4
LR scheduler  : ReduceLROnPlateau (patience=3)

Dummy loss    : 2.7446

✓ Loss + optimizer ready


###Build model for 51 users (EyeDentify)

In [8]:
# build model for 51 users (EyeDentify — Kaggle)
model_51 = EyeMotionID(n_users=51).to(device)

total_51     = sum(p.numel() for p in model_51.parameters())
trainable_51 = sum(p.numel() for p in model_51.parameters()
                   if p.requires_grad)

print("=== EyeMotionID Model Comparison ===\n")
print(f"{'Config':<20} {'15 users':<15} {'51 users'}")
print("-" * 50)
print(f"{'Total params':<20} {sum(p.numel() for p in model_15.parameters()):,}     {total_51:,}")
print(f"{'Trainable':<20} {sum(p.numel() for p in model_15.parameters() if p.requires_grad):,}       {trainable_51:,}")
print(f"{'Output classes':<20} {'15':<15} {'51'}")
print()
print("→ Use model_15 on Colab/MPIIGaze  (faster, less GPU)")
print("→ Use model_51 on Kaggle/EyeDentify (full experiment)")
print("\n✓ Both models ready")

=== EyeMotionID Model Comparison ===

Config               15 users        51 users
--------------------------------------------------
Total params         12,495,183     12,504,435
Trainable            1,318,671       1,327,923
Output classes       15              51

→ Use model_15 on Colab/MPIIGaze  (faster, less GPU)
→ Use model_51 on Kaggle/EyeDentify (full experiment)

✓ Both models ready


###Save model architecture

In [9]:
# save model_15 architecture
path_15 = os.path.join(MODELS, "eyemotionid_15users_architecture.pth")
torch.save({
    'model_state_dict' : model_15.state_dict(),
    'n_users'          : 15,
    'seq_len'          : 10,
    'hidden_size'      : 256,
    'num_layers'       : 2,
    'dropout'          : 0.5,
}, path_15)
print(f"✓ model_15 saved → {path_15}")

# save model_51 architecture
path_51 = os.path.join(MODELS, "eyemotionid_51users_architecture.pth")
torch.save({
    'model_state_dict' : model_51.state_dict(),
    'n_users'          : 51,
    'seq_len'          : 10,
    'hidden_size'      : 256,
    'num_layers'       : 2,
    'dropout'          : 0.5,
}, path_51)
print(f"✓ model_51 saved → {path_51}")
print("\n✓ Both architectures saved for Day 17 training")

✓ model_15 saved → /content/drive/MyDrive/EyeMotionID/models/checkpoints/eyemotionid_15users_architecture.pth
✓ model_51 saved → /content/drive/MyDrive/EyeMotionID/models/checkpoints/eyemotionid_51users_architecture.pth

✓ Both architectures saved for Day 17 training


###Summary

In [11]:
print("=" * 50)
print("   DAY 16 — LSTM ARCHITECTURE SUMMARY")
print("=" * 50)
print()
print("Model: EyeMotionID (CNN + LSTM)")
print()
print("Architecture:")
print("  Input   → (batch, 10, 3, 224, 224)")
print("  CNN     → ResNet18 frozen → 512-dim features")
print("  LSTM    → 2 layers, hidden=256")
print("  Dropout → 0.5")
print("  Output  → n_users classes")
print()
print("Models built:")
print("  ✓ EyeMotionID (15 users) — for MPIIGaze/Colab")
print("  ✓ EyeMotionID (51 users) — for EyeDentify/Kaggle")
print()
print("Saved:")
print("  ✓ eyemotionid_15users_architecture.pth")
print("  ✓ eyemotionid_51users_architecture.pth")
print()
print("Training config:")
print("  ✓ CrossEntropyLoss")
print("  ✓ Adam optimizer (lr=0.001)")
print("  ✓ ReduceLROnPlateau scheduler")
print()
print("=" * 50)

   DAY 16 — LSTM ARCHITECTURE SUMMARY

Model: EyeMotionID (CNN + LSTM)

Architecture:
  Input   → (batch, 10, 3, 224, 224)
  CNN     → ResNet18 frozen → 512-dim features
  LSTM    → 2 layers, hidden=256
  Dropout → 0.5
  Output  → n_users classes

Models built:
  ✓ EyeMotionID (15 users) — for MPIIGaze/Colab
  ✓ EyeMotionID (51 users) — for EyeDentify/Kaggle

Saved:
  ✓ eyemotionid_15users_architecture.pth
  ✓ eyemotionid_51users_architecture.pth

Training config:
  ✓ CrossEntropyLoss
  ✓ Adam optimizer (lr=0.001)
  ✓ ReduceLROnPlateau scheduler

